# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [7]:
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q pypdf pymupdf
!pip install -q sentence-transformers
!pip install -q chromadb
!pip install -q scikit-learn

In [8]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

In [10]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    # TODO: Implement PDF ingestion using PyPDFLoader.
    # Load all pdf files recursively from pdf_directory (using Path(pdf_directory).glob('**/*.pdf')).
    # For each page document loaded, add metadata fields:
    # - 'source_file': the filename of the PDF file
    # - 'file_type': 'pdf'
    # Collect all loaded documents in a list and return them.
    # Hint: Use PyPDFLoader(str(pdf_file)).load()
    documents = []

    pdf_files = Path(pdf_directory).glob("**/*.pdf")

    for pdf_file in pdf_files:
      loader = PyPDFLoader(str(pdf_file))
      pages = loader.load()

      for page in pages:
        page.metadata["source_file"] = pdf_file.name
        page.metadata["file_type"] = "pdf"
        documents.append(page)

    print(f"Loaded {len(documents)} pages from PDFs")

    return documents


In [11]:
!mkdir -p pdfs
!wget -O pdfs/sample.pdf https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf

--2026-06-11 17:00:41--  https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf
Resolving www.w3.org (www.w3.org)... 104.18.23.19, 104.18.22.19, 2606:4700::6812:1613, ...
Connecting to www.w3.org (www.w3.org)|104.18.23.19|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13264 (13K) [application/pdf]
Saving to: ‘pdfs/sample.pdf’

pdfs/sample.pdf     100%[===================>]  12.95K  --.-KB/s    in 0s      

2026-06-11 17:00:41 (27.6 MB/s) - ‘pdfs/sample.pdf’ saved [13264/13264]



In [13]:
all_pdf_documents = process_all_pdfs("pdfs")

Loaded 1 pages from PDFs


# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    # TODO: Implement text splitting using RecursiveCharacterTextSplitter.
    # You need to use RecursiveCharacterTextSplitter with:
    # - chunk_size=chunk_size of your choice
    # - chunk_overlap=chunk_overlap (recommended 20% of chunk_size)
    # - length_function=len
    # - separators=['\n\n', '\n', ' ', '']
    # Print how many documents were split into how many chunks. (Tip: do not use too big files it will it time)
    # Return the split documents list.
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(chunks)} chunks")

    return chunks


In [17]:
chunks=split_documents(all_pdf_documents)
chunks
print(chunks[:1])

Split 1 documents into 1 chunks
[Document(metadata={'producer': 'OpenOffice.org 2.1', 'creator': 'Writer', 'creationdate': '2007-02-23T17:56:37+02:00', 'author': 'Evangelos Vlachogiannis', 'source': 'pdfs/sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample.pdf', 'file_type': 'pdf'}, page_content='Dummy PDF file')]


# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [18]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        # TODO: Load the SentenceTransformer model using self.model_name.
        # Print a message stating the model name is being loaded.
        # Set self.model to the loaded SentenceTransformer.
        # Print a message indicating successful loading and print the embedding dimension.
        print(f"Loading embedding model: {self.model_name}")

        self.model = SentenceTransformer(self.model_name)

        print("Model loaded successfully")

        print(
          f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
        )

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        # TODO: Generate embeddings for the given list of texts using the model.
        # Make sure to handle cases where self.model is None.
        # Use self.model.encode(texts, show_progress_bar=True).
        # Return the resulting numpy array of embeddings.
        if self.model is None:
            raise ValueError("Model not loaded")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        return np.array(embeddings)


In [26]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully
Embedding dimension: 384


/tmp/ipykernel_7769/3831160991.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


In [27]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(1, 384)



# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [29]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        # TODO: Create the persist directory if it doesn't exist.
        # Initialize chromadb.PersistentClient with path=self.persist_directory.
        # Get or create a collection using self.client.get_or_create_collection
        # with self.collection_name and description metadata.
        os.makedirs(
        self.persist_directory,
        exist_ok=True
        )

        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embeddings"
            }
        )

        print(f"Collection '{self.collection_name}' initialized")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        # TODO: Add documents to ChromaDB collection.
        # 1. Check if the number of documents matches the number of embeddings.
        # 2. For each document, generate a unique ID using uuid.uuid4().hex[:8]
        # 3. Construct metadata dict including original document metadata, doc_index, and content_length.
        # 4. Extract document content string.
        # 5. Convert embedding to a list using .tolist().
        # 6. Add ids, embeddings, metadatas, and documents to self.collection.
        if len(documents) != len(embeddings):
          raise ValueError(
            "Documents and embeddings count mismatch"
        )

        ids = []
        metadatas = []
        docs = []
        embedding_list = []

        for idx, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            ids.append(uuid.uuid4().hex[:8])

            metadata = {
                **doc.metadata,
                "doc_index": idx,
                "content_length": len(doc.page_content)
            }

            metadatas.append(metadata)
            docs.append(doc.page_content)
            embedding_list.append(
                embedding.tolist()
            )

        self.collection.add(
            ids=ids,
            embeddings=embedding_list,
            metadatas=metadatas,
            documents=docs
        )

        print(
            f"Added {len(documents)} documents to vector store"
        )


In [30]:
chunks

[Document(metadata={'producer': 'OpenOffice.org 2.1', 'creator': 'Writer', 'creationdate': '2007-02-23T17:56:37+02:00', 'author': 'Evangelos Vlachogiannis', 'source': 'pdfs/sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample.pdf', 'file_type': 'pdf'}, page_content='Dummy PDF file')]

In [31]:
vectorstore = VectorStore()

Collection 'pdf_documents' initialized


In [32]:
vectorstore.add_documents(
    chunks,
    embeddings
)

Added 1 documents to vector store


In [33]:
print(
    vectorstore.collection.count()
)

1


## convert text to embeddings


In [34]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['Dummy PDF file']

In [35]:
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Added 1 documents to vector store


# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [36]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        # TODO: Retrieve relevant documents from the ChromaDB collection.
        # 1. Generate query embedding using self.embedding_manager.generate_embeddings.
        # 2. Query the vector store collection with the query embedding list, asking for top_k results.
        # 3. For each result, convert distance to similarity score (similarity_score = 1 - distance).
        # 4. Filter results that have similarity_score >= score_threshold.
        # 5. Return a list of dicts with keys: 'id', 'content', 'metadata', 'similarity_score', 'distance', 'rank'.
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )

        results = self.vector_store.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )

        retrieved_docs = []

        for i in range(len(results["ids"][0])):

            distance = results["distances"][0][i]

            similarity_score = 1 - distance

            if similarity_score >= score_threshold:

                retrieved_docs.append({
                    "id": results["ids"][0][i],
                    "content": results["documents"][0][i],
                    "metadata": results["metadatas"][0][i],
                    "similarity_score": similarity_score,
                    "distance": distance,
                    "rank": i + 1
                })

        return retrieved_docs


In [37]:
rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)

In [38]:
results = rag_retriever.retrieve(
    "dummy pdf",
    top_k=5
)

results

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': '75572a53',
  'content': 'Dummy PDF file',
  'metadata': {'page_label': '1',
   'content_length': 14,
   'doc_index': 0,
   'source_file': 'sample.pdf',
   'page': 0,
   'total_pages': 1,
   'file_type': 'pdf',
   'creator': 'Writer',
   'author': 'Evangelos Vlachogiannis',
   'source': 'pdfs/sample.pdf',
   'producer': 'OpenOffice.org 2.1',
   'creationdate': '2007-02-23T17:56:37+02:00'},
  'similarity_score': 0.8438258618116379,
  'distance': 0.15617413818836212,
  'rank': 1},
 {'id': 'd84d48dc',
  'content': 'Dummy PDF file',
  'metadata': {'doc_index': 0,
   'page_label': '1',
   'author': 'Evangelos Vlachogiannis',
   'source': 'pdfs/sample.pdf',
   'page': 0,
   'file_type': 'pdf',
   'total_pages': 1,
   'content_length': 14,
   'producer': 'OpenOffice.org 2.1',
   'source_file': 'sample.pdf',
   'creationdate': '2007-02-23T17:56:37+02:00',
   'creator': 'Writer'},
  'similarity_score': 0.8438258618116379,
  'distance': 0.15617413818836212,
  'rank': 2}]

In [40]:
rag_retriever.retrieve("dummy")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[]

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [41]:
def rag_simple(query,retriever,llm,top_k=3):
    # TODO: Implement a simple RAG pipeline using following steps which join above functions.
    # 1. Use the retriever to fetch top_k documents for the query.
    # 2. Join the content of the retrieved documents to form the context.
    # 3. Format a prompt instructing the LLM to use the context to answer the question.
    # 4. Call the LLM with the formatted prompt and return the string content of the response.
    docs = retriever.retrieve(
    query,
    top_k=top_k
    )

    context = "\n\n".join(
        [doc["content"] for doc in docs]
    )

    prompt = f"""
    Use the following context to answer the question.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content


In [45]:
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00


In [61]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY_HERE"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)


print("LLM Ready")

LLM Ready


In [62]:
answer = rag_simple(
    "three reasons for forgetting",
    rag_retriever,
    llm
)

print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 7.809665358s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url'

In [63]:
 answer=rag_simple("three reasons for forgetting",rag_retriever,llm)
 print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 16.214537104s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [64]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # TODO: Implement advanced RAG retrieval and generation.
    # 1. Retrieve top_k documents using score threshold min_score.
    # 2. Parse sources including source_file, page, similarity_score, and a content preview.
    # 3. Determine confidence (e.g. max similarity score of retrieved docs).
    # 4. Invoke LLM with formatted prompt combining context and query.
    # 5. Return dict containing 'answer', 'sources', 'confidence' (and 'context' if return_context is True).
    pass


In [65]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # TODO: Implement the AdvancedRAGPipeline query logic:
        # 1. Retrieve documents using self.retriever.
        # 2. Parse sources/metadata and construct the context.
        # 3. If stream=True, simulate streaming by printing the prompt in chunks.
        # 4. Generate the answer using self.llm.
        # 5. Construct citations list and append it to the answer.
        # 6. If summarize=True, call the LLM to get a 2-sentence summary of the answer.
        # 7. Append the question, answer, sources, and summary to self.history.
        # 8. Return dictionary containing question, answer (with citations), sources, summary, and history.
        pass


In [66]:
pipeline = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

result = pipeline.query(
    "three reasons for forgetting"
)

print(result)

None
